# PTB Diagnostic Database Analysis

## Setup

In [ ]:
PATH = "ptb-diagnostic-ecg-database-1.0.0"
ZIP = f"{PATH}.zip"
URL = f"https://www.physionet.org/static/published-projects/ptbdb/{ZIP}"

In [ ]:
import os
try:
    from google.colab import drive
    mount_point = os.path.join(os.path.sep, "content", "drive")
    drive.mount(mount_point)
    PATH = os.path.join(mount_point, "My Drive", PATH)
except ImportError:
    pass
SHELL_PATH = "\"" + PATH + "\""

In [ ]:
![ ! -d {SHELL_PATH} ] && wget -N -c {URL} && unzip -qq {ZIP}

## Content

List the contents of the main directory:

In [ ]:
sorted(os.listdir(PATH))

There are some general information and subfolders organised by patient.

List a subdirectory, e.g. `patient001`:

In [ ]:
import datetime
import humanize
import pandas as pd

patient_path = os.path.join(PATH, "patient001")
patient_file_info = [
    [
        file.name,
        humanize.naturalsize(file.stat().st_size),
        str(datetime.datetime.fromtimestamp(file.stat().st_mtime))
    ]
    for file in os.scandir(patient_path)
]
patient_file_info = sorted(patient_file_info)
patient_file_df = pd.DataFrame(
    patient_file_info,
    columns=["file", "size", "timestamp"]
)
patient_file_df

Files appear in groups of three, a `.dat`, a `.hea`, and a `.xyz`.
Looking at sizes:

- `.hea` are small around a few kB,
- `.xyz` are medium at few hundred kB,
- `.dat` are large around a few MB.

Note that **numberings are not consecutive** (0010, 0014, 0016) and there are **two endings** (`_re` and `lre`).

Try to read the `.hea`:

In [ ]:
hea_path = os.path.join(patient_path, "s0010_re.hea")
with open(hea_path) as hea_file:
    print(hea_file.read())

Luckily this is human-readable.

- The first part gives some technical data - note the lead identifiers and which file they are contained in.
- The second part gives clinical metadata!

There is also a "global" `RECORDS` file, let's read it.

In [ ]:
records_path = os.path.join(PATH, "RECORDS")
with open(records_path) as records_file:
    print(records_file.read())

Looks like a list of every ECG in the dataset, let's read it in a dataframe.

In [ ]:
records_df = pd.read_csv(records_path, header=None, names=["name"])
records_df["patient"] = records_df["name"].apply(lambda x: x.split("/")[0])
records_df["id"] = records_df["name"].apply(lambda x: x.split("/")[-1])
records_df["ending"] = records_df["id"].apply(lambda x: x[-3:])
records_df["id"] = records_df["id"].apply(lambda x: x[:-3])
records_df

The 3-character codes at the end of the naming scheme are not self-explanatory.

Maybe some statistics can help to understand them?

In [ ]:
records_df["ending"].value_counts().sort_index()

This is a good question for the customer, or ideally the person who collected the data.

**Learnings**

- Getting an elementary background in the topic pays off:
even ECG lead acronyms would have been difficult to interpret otherwise.
- Some dataset features may be obscure even to the best data scientist,
it usually is a good idea to ask the data provider for clarifications.

**Task 2.1**

What aspects of the dataset should be investigated?
Which statistics and plots would be interesting to understand it?

## Basic stats

Start with counting the number of patients in the dataset,
and how many files per patient are available.

In [ ]:
patient_dirs = sorted(x for x in os.listdir(PATH) if x.startswith("patient"))
len(patient_dirs)

The patients with the lowest and highest number are

In [ ]:
first_patient_string = patient_dirs[0]
last_patient_string = patient_dirs[-1]
print(first_patient_string, last_patient_string)

There are clearly some patients whose records are missing.
Their numbers are

In [ ]:
first_patient_number = int(first_patient_string[-3:])
last_patient_number = int(last_patient_string[-3:])
missing_patient_numbers = [
    n for n in range(first_patient_number, last_patient_number + 1)
    if f"patient{n:03}" not in patient_dirs
]
missing_patient_numbers

They do not really reveal a pattern,
it is reasonable to assume that they were discarded from the study for some reason
(e.g. corrupted data).

Next, it should be checked how many ECG recordings per patient are available.

In [ ]:
ecgs_per_patient = {}
for patient_dir in patient_dirs:
    patient_path_ = os.path.join(PATH, patient_dir)
    header_filenames = [filename for filename in os.listdir(patient_path_) if filename.endswith(".hea")]
    ecgs_per_patient[patient_dir] = len(header_filenames)

In [ ]:
import matplotlib.pyplot as plt
xs = ecgs_per_patient.values()
bins = max(xs) - min(xs) + 1
range_ = (min(xs) - 0.5, max(xs) + 0.5)
plt.hist(xs, range=range_, bins=bins)
plt.title("Number of ECG records per patient")
plt.xlabel("ECG records")
plt.ylabel("Patients")
plt.show()

The dataset description states that there are up to 5 recordings per patient.
The data seems to disagree.
A good check is to find out which data are responsible,
in this case the patients with 7 ECG recordings.

In [ ]:
patients_7 = [patient for patient in patient_dirs if ecgs_per_patient[patient] == 7]
patients_7

Now check the available header files.

In [ ]:
patient_7_path = os.path.join(PATH, patients_7[0])
sorted(filename for filename in os.listdir(patient_7_path) if filename.endswith(".hea"))

It is easy to check that this is a healthy control patient with ECGs from 1992 to 1997,
including one with unregistered date.
(What a dedication!)

Also note that the age is always registered to be the same...


**Learnings**

- Look for patterns in data organisation: it usually pays off.
- In case of inconsistencies, it is a good idea to explore in detail
and discuss them with data providers.

## Loading

Browsing the web,
it is easy to discover that the dataset has a relatively standard format
and there is a `python` library to read/write files.
It is called [`wfdb`](https://github.com/MIT-LCP/wfdb-python),
it has a documented [API](https://wfdb.readthedocs.io),
and indications on how to properly [cite the work](https://physionet.org/content/wfdb-python/4.0.0/).

Take the first record of the first patient and have a look.

In [ ]:
!pip install wfdb

In [ ]:
import wfdb
record_name = os.path.join(patient_path, "s0010_re")
signals, fields = wfdb.rdsamp(record_name)
print(signals)
print(fields)

`signals` is a matrix of floating-point numbers,
`fields` is a dictionary with metadata including clinical information.

The shape of `signals` is also interesting to look at:

In [ ]:
signals.shape

There are 38400 values for each of the 15 leads.

Now it makes sense to read the whole data
(and also to have a progress bar).

In [ ]:
from tqdm.auto import tqdm
all_signals = []
all_fields = []
for record_name_ in tqdm(records_df["name"]):
    record_path = os.path.join(PATH, record_name_)
    signals, fields = wfdb.rdsamp(record_path)
    all_signals.append(signals)
    all_fields.append(fields)
records_df["signals"] = all_signals
records_df["fields"] = all_fields
records_df

At this stage, the column `"fields"` contains a lot of semi-organised information;
it can be unpacked in further dataframe columns.

In [ ]:
fields = records_df["fields"][0].keys()
for field in fields:
    records_df[field] = records_df["fields"].apply(lambda x: x[field])
records_df.drop(columns=["fields"], inplace=True)
records_df

Much of the information is still packed in the `"comments"` column,
here is the example from the first record:

In [ ]:
records_df["comments"][0]

These are strings in the format `"key: values"`,
first find all possible keys.

In [ ]:
import itertools
all_comments = itertools.chain(*records_df["comments"])
all_comment_keys = sorted(set(x.split(":")[0] for x in all_comments))
all_comment_keys

Now unroll all information in columns of `records_df`,
and eliminate whitespaces at the head of the values string.

In [ ]:
def no_start_whitespace(x: str) -> str:
    return x[1:] if x.startswith(" ") else x
comment_dicts = [
    {comment.split(":")[0]: no_start_whitespace(comment.split(":")[1]) for comment in comments}
    for comments in records_df["comments"]
]
for comment_key in all_comment_keys:
    records_df[comment_key] = [
        comment_dict.get(comment_key, None)
        for comment_dict in comment_dicts
    ]
records_df.drop(columns=["comments"], inplace=True)
records_df.columns

## Features

**Task 2.2** What should we look at in this dataset?

There are tools to get automated reports.
Try to use `ydata_profiling` out of the box with data in the current format.

In [ ]:
!pip install ydata_profiling

In [ ]:
from ydata_profiling import ProfileReport
no_signals_df = records_df.drop(columns=["signals"])
profile = ProfileReport(no_signals_df, title="ECG dataset report draft")
profile.to_file("ecg_report_draft.html")

Observations:

- Some features are trivial, i.e. constant across the whole dataset.
- `ydata_profiling` reports missing values, but is not smart enough to understand that the string "n/a" means missing.
- Likewise, many features are strings which include units, and should be converted to floating point numbers.
- Dates have different formats and should be converted to machine-understandable formats.
- Some variables are actually lists of medical observations, and should be encoded differently.
- Variables that are largely missing are probably not very useful for machine learning (will inevitably lead to overfitting).

Some data wrangling is in order.
Start by eliminating trivial features.

In [ ]:
for column in records_df.columns:
    if column == "signals":
        continue
    try:
        column_uniques = records_df[column].unique()
    except TypeError:
        column_uniques = records_df[column].drop_duplicates()
    if len(column_uniques) == 1:
        print(f"{column} is always {column_uniques[0]}, deleting")
        records_df.drop(columns=[column], inplace=True)

Look at the value distribution for some features
to get more of a feeling for data quality.

First consider signal length as an indication of how uniform data collection was.

In [ ]:
records_df["sig_len"].value_counts().sort_index()

Signal length is mostly standard (38.4s, 115.2s, 120.012s) with a few exceptions.

Now look at a clinical label, in this case choose the localization of acute infarction.

In [ ]:
records_df['Acute infarction (localization)'].value_counts().sort_index()

Clearly even if the dataset is already very clean,
there are still typos:

- `"infero-latera"` = `"infero-lateral"`
- `"infero-poster-lateral"` = `"infero-postero-lateral"`

Often such basic checks are forgotten,
and a ML model has additional categories which are in reality the same as others.

Some distinctions may be unclear: `"unknown"` vs `"n/a"` vs `"no"`.
A reasonable way to proceed would be to take `"no"` as `"no infarction"`,
and treat `"unknown"` as `"n/a"`,
but it is best to discuss with the data provider!

Another clinical label that could be very interesting are additional diagnoses.

In [ ]:
additional_diagnoses = [
    sorted(no_start_whitespace(ad) for ad in ad_str.split(","))
    for ad_str in records_df["Additional diagnoses"]
]
sorted(set(itertools.chain(*additional_diagnoses)))

Lots of simple but very annoying complications arise:

- capitalisation,
- typos,
- different levels of detail,
- different names for the same conditions.

If these are injected in a ML model without preprocessing,
the model will not know the relationship among terms (e.g. all just "distinct" categories)
and will have a very hard time in figuring them out (large amounts of data needed).

To alleviate this problem,
use **ontologies** such as [ICD](https://icd.who.int/en) (Creative Commons)
or [SNOMED CT](https://www.snomed.org/snomed-ct/five-step-briefing) (commercial).
Note that typos have to be corrected manually if relevant.

## Possible tasks

It is very important to understand the Machine Learning tasks that can be performed with a given dataset,
and if the customer's requests can be satisfied.

**Task 3: what could we do with this dataset?**

Here the focus will be on _classification of ECGs in acute disease classes_,
which are the reason for admission to a hospital.

## Advanced stats

Now that some aspects of the data have been explored on a sample basis,
_demographics_ and _timeline_ should be explored more systematically.

In [ ]:
print(sorted(records_df["age"].unique()))

Age is encoded as strings, better transform it to numeric values.

In [ ]:
records_df["age"] = pd.to_numeric(records_df["age"], errors="coerce")

Look at the possible values for sex.

In [ ]:
records_df["sex"].value_counts()

There is one record with a blank string, set it to `"n/a"`.

In [ ]:
records_df.loc[records_df["sex"] == "", "sex"] = "n/a"
records_df["sex"].value_counts()

Look at recorded values for smoking habits (rename all features to lowercase).

In [ ]:
records_df.rename(columns={"Smoker": "smoker"}, inplace=True)
records_df["smoker"].value_counts()

Map `"unknown"` to `"n/a"`, such that only one undetermined value occurs.

In [ ]:
records_df.loc[records_df["smoker"] == "unknown", "smoker"] = "n/a"
records_df["smoker"].value_counts()

Explore if patients change age, sex, or smoking habits,
by checking if the values across ECG records for the same patient are always the same.

In [ ]:
patient_df = records_df[["patient", "age", "sex", "smoker"]].drop_duplicates()
patient_df["patient"].is_unique

Visualize information about age vs sex.

In [ ]:
import plotly.express as px
px.histogram(patient_df, x="age", color="sex", barmode="overlay")

Visualize smoking habits vs sex.

In [ ]:
px.histogram(patient_df, x="smoker", color="sex", barmode="overlay")

Dates need to be converted from strings to a format which enables calculations.

In [ ]:
def parse_date(datestr: str) -> datetime.date:
    try:
        if "/" in datestr:
            return datetime.datetime.strptime(datestr, "%d/%m/%Y")
        else:
            return datetime.datetime.strptime(datestr, "%d-%b-%y")
    except ValueError:
        return pd.NaT

Missing values are then easily checked.

In [ ]:
records_df["ecg_date"] = records_df["ECG date"].apply(parse_date)
records_df["admission_date"] = records_df["Admission date"].apply(parse_date)
records_df["infarction_date"] = records_df["Infarction date"].apply(parse_date)
records_df[["ecg_date", "admission_date", "infarction_date"]].isna().sum(axis=0)

That `infarction_date` be missing for a large number of records is expected:
not every patient has had an infarction.

It is instead a good idea to see if there are obvious patterns in the few records
which are missing a substantial information that should always be there,
such as ECG date.

In [ ]:
records_df[records_df["ecg_date"].isna()]

No obvious trend can be seen from these records.

Plot the events along a timeline for all patients.

In [ ]:
from typing import Sequence
plt_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
def eventplot(data: pd.DataFrame, fields: Sequence[str], title: str = ""):
    plt.figure(figsize=(10, len(fields)))
    positions = [[t for t in data[field] if t == t] for field in fields]
    plt.eventplot(positions=positions, color=plt_colors[:len(fields)])
    plt.yticks(ticks=tuple(range(len(fields))), labels=fields)
    plt.title(title)
    plt.tight_layout()
    plt.show()

In [ ]:
eventplot(records_df, ("ecg_date", "admission_date", "infarction_date"))

The last infarction date is later than the last ECG date?
Investigate this example further.

In [ ]:
records_df.loc[records_df["infarction_date"].argmax()]

The patient already had an infarction in '91, so data are consistent.

Plot all events for a few patients with >1 ECG.

In [ ]:
counter = 0
for patient, patient_df_ in records_df.groupby("patient"):
    if len(patient_df_) > 1:
        eventplot(patient_df_, ("ecg_date", "admission_date", "infarction_date"))
        print(patient_df_["ecg_date"])
        counter += 1
    if counter >= 5:
        break

Rename the diagnosis column and check if it is unique for all patients.

In [ ]:
records_df.rename(columns={"Reason for admission": "diagnosis"}, inplace=True)
patient_df = records_df[["patient", "age", "sex", "smoker", "diagnosis"]].drop_duplicates()
patient_df["patient"].is_unique

Obtain some simple statistics of diagnoses by patient.

In [ ]:
diagnosis_counts = patient_df["diagnosis"].value_counts()
diagnosis_counts

**Task 4: what observations can be made from this distribution?**

_The dataset is clearly not balanced, nor is the distribution representative of the general population._

This is common in medical datasets.
Very often pathologic cases in general are over-represented due to sampling at the point-of-care,
even if healthy cases would be relatively easy to obtain.
More rare pathologies often have very little data even in large datasets,
so one needs to deal with imbalanced datasets.

Notes:

- Myocardial infarction is well represented.
- We have enough healthy controls, but clearly less than in the general population! Metrics will have to take this into account.
- Diseases with one patient cannot be trained and evaluated on different subjects.
- Diseases with only a handful of subjects will have very poor statistics, which makes them not very meaningful.

## Examples

A last very important step of exploration is to actually plot the signals for a few cases.

In [ ]:
record = wfdb.rdrecord(record_name)
wfdb.plot.plot_wfdb(record, figsize=(16, 16))

Different heartbeat patterns for each lead can be observed.
It is also easy to note there can be baseline drifts
(usually attributed to patient movements).